# Level 1 - Detection

Binary: 0 = authentic, 1 = manipulated (fully AI-generated or locally edited).
Metric is F1 on the positive class.

Three things drive the setup here:

- All-ones scores F1 = 0.800, since positives are 2/3 of the data. That is the floor.
- The F1-optimal threshold sits well below 0.5, so it gets fitted on out-of-sample
  predictions rather than left at 0.5.
- Images are 640x640 PNG and about a third of the positives are small local edits.
  Downscaling wipes the resampling traces that give those away, so everything runs
  at native resolution with flips and rotations only.

Runtime: T4 x2 with internet on for the pretrained weights. Not the P100, the torch
2.10 build here has no sm_60 kernels. Holdout takes 75-85 min.

## Config

In [ ]:
class CFG:
    seed = 42
    n_folds = 5

    # Encoder. Has to be a CNN if eval_size != train_size, since ViT and Swin have
    # fixed position embeddings and break on a resolution change.
    model_name = "tf_efficientnetv2_s.in21k_ft_in1k"
    pretrained = True
    drop_rate = 0.3
    drop_path_rate = 0.2

    # 640 is native, so no cropping. A smaller train_size buys augmentation but can
    # crop the edited region out of a Cat 2 image, which is label noise.
    train_size = 640
    eval_size = 640

    # Val AUC hit 1.0000 at epoch 1 and never moved through epoch 7, so 8 epochs was
    # about 4x more than needed. 3 leaves margin past saturation and keeps 5-fold cv
    # at ~1.5 h instead of 3.7 h.
    epochs = 3
    batch_size = 8
    grad_accum = 4         # effective batch = 32
    eval_batch_size = 16
    lr = 2e-4
    weight_decay = 1e-2
    warmup_frac = 0.1
    max_grad_norm = 1.0
    num_workers = 4
    amp = True

    out_dir = "/kaggle/working"


# Positive class is `manipulated`. Named once, used everywhere.
POSITIVE_CLASS = 1


## Imports

In [ ]:
import gc
import os
import random
import time

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

# cv2 spawns its own thread pool which fights the DataLoader workers.
cv2.setNumThreads(0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ENABLED = CFG.amp and DEVICE == "cuda"


def autocast():
    return torch.amp.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)
# benchmark=True picks the fastest conv kernels. With the seeding above, runs match
# closely but not bit-exactly. Full determinism roughly halves throughput.
torch.backends.cudnn.benchmark = True

os.makedirs(CFG.out_dir, exist_ok=True)


## Data paths

Same file works on Kaggle and locally. Watch the column name: ground_truth.csv
calls it `class`, the submission wants `classe`.

In [ ]:
DATA_ROOT = "/kaggle/input/gensivana-real-or-fake-level-1-detection"
TEST_DIR = os.path.join(DATA_ROOT, "test", "images")


def load_ground_truth(split: str) -> pd.DataFrame:
    """Load a split's ground_truth.csv, normalising the label column to `label`."""
    df = pd.read_csv(os.path.join(DATA_ROOT, split, "ground_truth.csv"))
    # Provided files use `class`, the submission wants `classe`. Accept either.
    label_col = next((c for c in ("class", "classe", "label") if c in df.columns), None)
    if label_col is None:
        raise KeyError(f"No label column in {split}/ground_truth.csv: {list(df.columns)}")
    df = df.rename(columns={label_col: "label"})
    df["label"] = df["label"].astype(int)
    df["dir"] = os.path.join(DATA_ROOT, split, "images")
    return df[["image_id", "label", "dir"]]


train_df = load_ground_truth("train")
val_df = load_ground_truth("val")

sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
test_df = pd.DataFrame({"image_id": sample_sub["image_id"].tolist(), "dir": TEST_DIR})

print(f"train {len(train_df):>5}   {dict(train_df.label.value_counts().sort_index())}")
print(f"val   {len(val_df):>5}   {dict(val_df.label.value_counts().sort_index())}")
print(f"test  {len(test_df):>5}")

# Fail loudly now rather than after an hour of training.
assert train_df.image_id.is_unique and val_df.image_id.is_unique
assert set(train_df.image_id) & set(val_df.image_id) == set(), "train/val leak"


## Baseline

Printed every run so a 0.79 never looks like progress.

In [ ]:
def f1_of_constant_prediction(y_true: np.ndarray) -> float:
    """F1 obtained by predicting the positive class for every sample."""
    return f1_score(y_true, np.ones_like(y_true), pos_label=POSITIVE_CLASS,
                    zero_division=0)


ALL_ONES_F1 = f1_of_constant_prediction(val_df.label.values)
pos_rate = float((val_df.label.values == POSITIVE_CLASS).mean())

print(f"val positive rate      : {pos_rate:.4f}")
print(f"all-ones baseline F1   : {ALL_ONES_F1:.4f}   <-- must beat this")
print(f"all-ones balanced acc  : 0.5000")

## Dataset

No resizing anywhere. Augmentation is D4 only, the 8 flip/rotation combinations, which
permute pixels without resampling them. No colour jitter or blur, those would destroy
the signal being detected.

Images come out as float in [0, 1], with normalisation done inside the model so
training and inference cannot drift apart.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, size: int, train: bool):
        self.ids = df.image_id.tolist()
        self.dirs = df.dir.tolist()
        self.labels = df.label.tolist() if "label" in df.columns else None
        self.size = size
        self.train = train

    def __len__(self) -> int:
        return len(self.ids)

    def _read(self, idx: int) -> np.ndarray:
        path = os.path.join(self.dirs[idx], self.ids[idx])
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"failed to read {path}")
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx: int):
        img = self._read(idx)
        h, w = img.shape[:2]

        if self.train:
            # Random crop only when the target is smaller than the source.
            if self.size < min(h, w):
                y = random.randint(0, h - self.size)
                x = random.randint(0, w - self.size)
                img = img[y:y + self.size, x:x + self.size]
            # D4 is resize-free, so no interpolation artefacts.
            k = random.randint(0, 3)
            if k:
                img = np.rot90(img, k)
            if random.random() < 0.5:
                img = img[:, ::-1]
        elif self.size < min(h, w):
            # Deterministic centre crop for evaluation.
            y = (h - self.size) // 2
            x = (w - self.size) // 2
            img = img[y:y + self.size, x:x + self.size]

        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        tensor = torch.from_numpy(img).float().div_(255.0)

        if self.labels is None:
            return tensor
        return tensor, torch.tensor(self.labels[idx], dtype=torch.float32)


def make_loader(df: pd.DataFrame, size: int, train: bool, batch_size: int) -> DataLoader:
    return DataLoader(
        ImageDataset(df, size, train),
        batch_size=batch_size,
        shuffle=train,
        drop_last=train,
        num_workers=CFG.num_workers,
        pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )

## Model

A timm encoder with a single-logit head. Normalisation lives inside the module so
training and inference cannot drift apart.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class DetectorNet(nn.Module):
    """Backbone wrapper that owns preprocessing so train and inference stay consistent."""

    def __init__(self, cfg=CFG, pos_rate=None):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.model_name,
            pretrained=cfg.pretrained,
            num_classes=1,
            in_chans=3,
            drop_rate=cfg.drop_rate,
            drop_path_rate=cfg.drop_path_rate,
        )

        # A fresh head emits logits near +/-8 at 640px, so the first steps go into
        # collapsing a loss of ~8 instead of learning (step 0 was 8.07 against the
        # 0.69 you would expect). Zeroing the weight and setting the bias to the
        # class prior starts it at ~0.64 with no early fp16 instability.
        if pos_rate is not None:
            head = self.backbone.get_classifier()
            p = float(np.clip(pos_rate, 1e-4, 1 - 1e-4))
            nn.init.zeros_(head.weight)
            if head.bias is not None:
                nn.init.constant_(head.bias, float(np.log(p / (1 - p))))

        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: raw RGB in [0, 1], shape (B, 3, H, W). Returns logits, shape (B,)."""
        return self.backbone((x - self.mean) / self.std).squeeze(1)


## Threshold selection

precision_recall_curve enumerates every achievable operating point, so there is no
grid to miss the optimum.

In [ ]:
def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[float, float]:
    """Return (threshold, F1) maximising F1 over all achievable operating points."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    # precision/recall have one more element than thresholds; drop the trailing
    # (recall=0, precision=1) sentinel which has no corresponding threshold.
    denom = precision[:-1] + recall[:-1]
    f1 = np.where(denom > 0, 2 * precision[:-1] * recall[:-1] / np.maximum(denom, 1e-12), 0.0)
    if len(f1) == 0:
        return 0.5, 0.0
    best_f1 = float(f1.max())
    # Several thresholds often tie at the optimum; take the middle one rather than
    # argmax's first, which sits at the very edge of the tied range.
    tied = np.flatnonzero(np.isclose(f1, best_f1, rtol=0, atol=1e-12))
    thr = float(thresholds[tied[len(tied) // 2]])

    # precision_recall_curve only proposes observed scores, so thr lands exactly on
    # a training point: with clean separation that is the lowest positive, a hair
    # above the highest negative. Slide it into the middle of the empty gap below.
    # Nothing observed lies in between so predictions here are identical, but a test
    # point landing slightly off gets the most margin. Midpoint in log-odds, since
    # these probabilities pile up near 0 and 1.
    below = y_prob[y_prob < thr]
    if below.size:
        eps = 1e-12

        def logit(p):
            p = np.clip(p, eps, 1 - eps)
            return float(np.log(p / (1 - p)))

        thr = float(1.0 / (1.0 + np.exp(-0.5 * (logit(float(below.max())) + logit(thr)))))
    return thr, best_f1


def evaluate(y_true: np.ndarray, y_prob: np.ndarray, threshold: float | None = None) -> dict:
    """Metric report. If threshold is None, the F1-optimal one is fitted."""
    fitted_thr, _ = best_f1_threshold(y_true, y_prob)
    thr = fitted_thr if threshold is None else threshold
    y_pred = (y_prob >= thr).astype(int)
    return {
        "auc": float(roc_auc_score(y_true, y_prob)),
        "threshold": float(thr),
        "f1": float(f1_score(y_true, y_pred, pos_label=POSITIVE_CLASS, zero_division=0)),
        "pos_rate": float(y_pred.mean()),
    }


def format_metrics(m: dict) -> str:
    return (f"auc {m['auc']:.4f} | thr {m['threshold']:.3f} "
            f"| f1 {m['f1']:.4f} | pos_rate {m['pos_rate']:.3f}")


## Train / predict

In [ ]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> np.ndarray:
    """Return sigmoid probabilities, one per image."""
    model.eval()
    probs = []
    for batch in loader:
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        x = x.to(DEVICE, non_blocking=True)
        with autocast():
            p = torch.sigmoid(model(x).float())
        probs.append(p.cpu().numpy())
    return np.concatenate(probs)


def train_one_fold(fold_train: pd.DataFrame, fold_valid: pd.DataFrame, tag: str) -> dict:
    """Train a single model. Returns val probs, test probs and metrics."""
    print(f"\n[{tag}] train {len(fold_train)}  valid {len(fold_valid)}")

    train_loader = make_loader(fold_train, CFG.train_size, True, CFG.batch_size)
    valid_loader = make_loader(fold_valid, CFG.eval_size, False, CFG.eval_batch_size)

    pos_rate = float((fold_train.label.values == POSITIVE_CLASS).mean())
    model = DetectorNet(CFG, pos_rate=pos_rate).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr,
                                  weight_decay=CFG.weight_decay)

    steps_per_epoch = max(1, len(train_loader) // CFG.grad_accum)
    total_steps = steps_per_epoch * CFG.epochs
    warmup_steps = max(2, int(total_steps * CFG.warmup_frac))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=CFG.lr,
        total_steps=total_steps,
        pct_start=warmup_steps / total_steps,
        anneal_strategy="cos",
    )
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    criterion = nn.BCEWithLogitsLoss()

    # Select on AUC, which is threshold-free. Selecting on F1 would tie the
    # checkpoint choice to a threshold that is not fitted yet.
    best_auc, best_state = -1.0, None

    for epoch in range(CFG.epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running, seen, t0 = 0.0, 0, time.time()

        for step, (x, y) in enumerate(train_loader):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast():
                loss = criterion(model(x), y)

            scaler.scale(loss / CFG.grad_accum).backward()

            if (step + 1) % CFG.grad_accum == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                # GradScaler skips optimizer.step() on inf/nan grads, which it does
                # on the first step at the initial scale of 65536 before halving it.
                # Advancing the schedule for an update that never happened is wrong
                # and triggers the "step() before optimizer.step()" warning.
                stepped = scaler.get_scale() >= scale_before
                if stepped and scheduler.last_epoch < total_steps - 1:
                    scheduler.step()

            running += loss.item() * x.size(0)
            seen += x.size(0)

        val_probs = predict(model, valid_loader)
        metrics = evaluate(fold_valid.label.values, val_probs)
        print(f"  ep{epoch} {time.time() - t0:.0f}s | loss {running / max(seen, 1):.4f} "
              f"| {format_metrics(metrics)}")

        if metrics["auc"] > best_auc:
            best_auc = metrics["auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Reload the best epoch before producing the predictions actually used.
    model.load_state_dict(best_state)

    test_loader = make_loader(test_df, CFG.eval_size, False, CFG.eval_batch_size)
    val_probs = predict(model, valid_loader)
    test_probs = predict(model, test_loader)
    final_metrics = evaluate(fold_valid.label.values, val_probs)
    print(f"[{tag}] best epoch: {format_metrics(final_metrics)}")

    del model, best_state
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "tag": tag,
        "val_labels": fold_valid.label.values,
        "val_probs": val_probs,
        "test_probs": test_probs,
        "metrics": final_metrics,
    }


## Run

5-fold StratifiedKFold over train and val pooled. That gives more data per model than
a fixed holdout, a 5-model ensemble, and an 8835-image out-of-fold set to fit the
threshold on. The 978-image holdout saturated early and stopped being able to rank
models at all.

In [ ]:
results = []
pool = pd.concat([train_df, val_df], ignore_index=True)
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)

for fold, (tr_idx, va_idx) in enumerate(skf.split(pool, pool.label)):
    results.append(train_one_fold(
        pool.iloc[tr_idx].reset_index(drop=True),
        pool.iloc[va_idx].reset_index(drop=True),
        f"fold{fold}",
    ))

print(f"\ntrained {len(results)} models")


## Fit the threshold out-of-sample

Every prediction here comes from a model that never saw the image. In cv that covers
all 8835 labelled images, which makes the threshold far steadier than fitting it on
978 holdout images.

In [ ]:
oof_labels = np.concatenate([r["val_labels"] for r in results])
oof_probs = np.concatenate([r["val_probs"] for r in results])

# Fold test predictions are averaged; each fold is an equally valid estimator.
test_probs = np.mean([r["test_probs"] for r in results], axis=0)

THRESHOLD, _ = best_f1_threshold(oof_labels, oof_probs)
oof_metrics = evaluate(oof_labels, oof_probs, threshold=THRESHOLD)

# Count misclassifications directly. (1-F1)*n is only a fair proxy near F1 = 1 and
# still understates by ~4/3, since F1's denominator double-counts TPs.
pred = (oof_probs >= THRESHOLD).astype(int)
errors = int((pred != oof_labels).sum())
fp = int(((pred == 1) & (oof_labels == 0)).sum())
fn = int(((pred == 0) & (oof_labels == POSITIVE_CLASS)).sum())

oof_allones_f1 = f1_of_constant_prediction(oof_labels)

print(f"OOF n = {len(oof_labels)}")
print(f"  {format_metrics(oof_metrics)}")
print(f"  errors {errors} of {len(oof_labels)}  (FP {fp}, FN {fn})")
print(f"\nchosen threshold         : {THRESHOLD:.4f}")
print(f"OOF F1                   : {oof_metrics['f1']:.4f}")
print(f"all-ones baseline F1     : {oof_allones_f1:.4f}")
print(f"improvement over baseline: {oof_metrics['f1'] - oof_allones_f1:+.4f}")


### Threshold sensitivity

A sharp peak means the threshold is fitted to the OOF set and will not transfer. A
broad plateau means it is safe. If it comes out sharp, take the middle of the plateau
instead of the argmax.

In [ ]:
grid = np.linspace(0.02, 0.98, 49)
rows = []
for t in grid:
    pred = (oof_probs >= t).astype(int)
    rows.append({
        "threshold": round(float(t), 3),
        "f1": round(float(f1_score(oof_labels, pred, pos_label=POSITIVE_CLASS,
                                   zero_division=0)), 4),
        "pos_rate": round(float(pred.mean()), 3),
    })
sens = pd.DataFrame(rows)
top = sens.sort_values("f1", ascending=False).head(12).sort_values("threshold")
print("top-12 thresholds by OOF F1:")
print(top.to_string(index=False))

plateau = sens[sens.f1 >= oof_metrics["f1"] - 0.002]
print(f"\nthresholds within 0.002 of the best: "
      f"[{plateau.threshold.min():.3f}, {plateau.threshold.max():.3f}] "
      f"({len(plateau)} of {len(sens)} grid points)")

## Build the submission

Worth keeping the validator: a malformed file is rejected for free, but a well-formed
wrong one costs one of the five daily attempts. Column is `classe`.

In [ ]:
def build_submission(ids: list[str], probs: np.ndarray, threshold: float) -> pd.DataFrame:
    return pd.DataFrame({"image_id": ids, "classe": (probs >= threshold).astype(int)})


def validate_submission(sub: pd.DataFrame, reference: pd.DataFrame) -> None:
    """Raise on anything the grader would reject."""
    assert list(sub.columns) == ["image_id", "classe"], f"bad columns: {list(sub.columns)}"
    assert len(sub) == len(reference), f"expected {len(reference)} rows, got {len(sub)}"
    assert sub.image_id.is_unique, "duplicate image_id"
    assert set(sub.image_id) == set(reference.image_id), "image_id set does not match test/"
    assert sub.classe.dtype.kind in "iu", f"classe must be integer, got {sub.classe.dtype}"
    assert set(sub.classe.unique()) <= {0, 1}, f"classe outside {{0,1}}: {sub.classe.unique()}"
    assert sub.notna().all().all(), "null values present"
    print("submission validated OK")


submission = build_submission(test_df.image_id.tolist(), test_probs, THRESHOLD)
validate_submission(submission, sample_sub)

sub_path = os.path.join(CFG.out_dir, "submission.csv")
submission.to_csv(sub_path, index=False)

print(f"\nwrote {sub_path}")
print(f"predicted positive rate : {submission.classe.mean():.4f}")
print(f"OOF positive rate       : {(oof_labels == POSITIVE_CLASS).mean():.4f}")


## Notes

5 submissions a day, 2 count at the end. Decide from OOF F1, not the public board:
993 test images put the public standard error near +/-0.015, which is wider than
most of what is left to gain.